<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/elliptic-pde/dtb_elliptic_two_frequency_colab_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTB vs. ordinary Deep Ritz vs. Fourier Ritz
## A two-frequency Poisson experiment on $(-1,1)^2$

This notebook asks whether outer Deep Tangent Bundle (DTB) updates adapt a small neural tangent space so that it recovers a low-amplitude but high-frequency component.

We solve

$$
\Omega=(-1,1)^2,\qquad -\Delta u=f\ \text{in }\Omega,\qquad u=0\ \text{on }\partial\Omega,
$$

with manufactured solution

$$
u_*(x,y)=\underbrace{\sin(\pi x)\sin(\pi y)}_{\phi_{11}(x,y)}
+0.1\underbrace{\sin(7\pi x)\sin(5\pi y)}_{\phi_{75}(x,y)}.
$$

Because $-\Delta\phi_{11}=2\pi^2\phi_{11}$ and $-\Delta\phi_{75}=74\pi^2\phi_{75}$,

$$
f(x,y)=2\pi^2\phi_{11}(x,y)+7.4\pi^2\phi_{75}(x,y).
$$

The three methods are:

1. frozen and adaptive DTB--Ritz at tangent rank $r$;
2. ordinary Deep Ritz, which optimizes every network parameter;
3. a fixed Fourier/Dirichlet Ritz space ordered by Laplacian eigenvalue.


## 1. Colab and repository setup

Google Colab already includes PyTorch, NumPy, and Matplotlib, so this experiment does not require a separate solver package. The cell below clones the elliptic-pde branch only when the shared utility file is not already present.


In [ ]:
# ============================================================
# COLAB / REPOSITORY SETUP
# ============================================================
import os
import subprocess
import sys
import time
from pathlib import Path

if Path("DTB_elliptic_utils.py").exists():
    # This path is used when the notebook is launched from a local clone.
    ROOT = Path.cwd()
else:
    # A fresh Colab runtime starts in /content and needs the repository.
    ROOT = Path("/content/dtb-colab-experiments")
    if not ROOT.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", "elliptic-pde",
                "https://github.com/sun-mengwei/dtb-colab-experiments.git",
                str(ROOT),
            ],
            check=True,
        )
    os.chdir(ROOT)

print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## 2. Imports and shared DTB machinery

Reusable operations remain in DTB_elliptic_utils.py. The notebook adds only experiment-specific problem definitions, diagnostics, runners, and plots.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import math
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.func import jacrev, jvp, vmap

from DTB_elliptic_utils import (
    MLP,
    assemble_ritz_system,
    box_boundary_factor,
    envelope_gradient,
    expand_direction,
    flatten_parameters,
    make_box_trial,
    manufactured_forcing,
    matrix_ritz_energy,
    sample_box,
    sample_box_boundary,
    select_parameter_indices,
    selected_tangent_basis,
    solve_ritz_system,
    tangent_values_and_gradients,
)

torch.set_default_dtype(torch.float64)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 3. Shared experiment configuration

This is the control panel for all three methods. The normal Colab run uses the full values from the experiment specification. Setting the environment variable DTB_NOTEBOOK_SMOKE=1 activates a tiny configuration used only for quick execution checks.


In [ ]:
# ============================================================
# SHARED EXPERIMENT CONFIGURATION
# ============================================================
SMOKE_TEST = os.environ.get("DTB_NOTEBOOK_SMOKE", "0") == "1"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# PDE and domain.
DIM = 2
VOLUME = 4.0                  # |(-1,1)^2|
DIFFUSION = None              # None means A = I.
REACTION = 0.0

# Shared quadrature and evaluation sets.
N_TRAIN = 256 if SMOKE_TEST else 4096
N_TEST = 512 if SMOKE_TEST else 16384
N_RESIDUAL_TEST = 64 if SMOKE_TEST else 2048
N_PLOT_SIDE = 41 if SMOKE_TEST else 161
TRAIN_SEED = 100
TEST_SEED = 999
MONITOR_EVERY = 1 if SMOKE_TEST else 2

# Reproducibility and robustness controls.
PRIMARY_SEED = 0
SEEDS = [0, 1] if SMOKE_TEST else [0, 1, 2, 3, 4]
PRIMARY_RANK = 8 if SMOKE_TEST else 32
TANGENT_RANKS = [8, 32, 64]
FOURIER_RANKS = [4, 8] if SMOKE_TEST else [8, 16, 32, 64]

# The requested DTB-only tangent-size comparison runs by default.
RUN_TANGENT_COMPARISON = True
RUN_SEED_SWEEP = False
RUN_ORACLE_FOURIER = True

DTB_OUTER_STEPS = 2 if SMOKE_TEST else 80
DTB_CHECKPOINTS = [0, DTB_OUTER_STEPS // 2, DTB_OUTER_STEPS]
DEEP_RITZ_STEPS = 3 if SMOKE_TEST else 5000

print(f"device={DEVICE}, smoke_test={SMOKE_TEST}")
print(f"N_train={N_TRAIN}, N_test={N_TEST}, primary rank={PRIMARY_RANK}")


## 4. Method-specific configurations

The neural methods use the same MLP width, depth, activation, initialization seed, hard boundary factor, and training points. Their iteration counts should not be interpreted as equal computational work, so wall-clock time is recorded separately.


In [ ]:
# ============================================================
# METHOD CONFIGURATIONS
# ============================================================
DTB_CFG = dict(
    network_width=32,
    network_depth=3,
    activation="tanh",
    tangent_rank=PRIMARY_RANK,
    tangent_seed=123,
    n_outer=DTB_OUTER_STEPS,
    ridge_relative=1.0e-8,
    chunk_size=64 if SMOKE_TEST else 256,
    fixed_step_size=1.0e-2,
    checkpoints=DTB_CHECKPOINTS,
)

DEEP_RITZ_CFG = dict(
    network_width=DTB_CFG["network_width"],
    network_depth=DTB_CFG["network_depth"],
    activation=DTB_CFG["activation"],
    optimizer="adam",
    learning_rate=1.0e-3,
    n_steps=DEEP_RITZ_STEPS,
    weight_decay=0.0,
    monitor_every=1 if SMOKE_TEST else 50,
    use_same_train_points=True,
    use_same_initial_seed=True,
)

FOURIER_CFG = dict(
    ranks=FOURIER_RANKS,
    basis="complete_dirichlet_sine",
    ordering="increasing_laplacian_eigenvalue",
    solve="ritz_linear_system",
    quadrature="same_sobol_points",
    ridge_relative=0.0,
)


## 5. PDE definition

For

$$
\phi_{11}=\sin(\pi x)\sin(\pi y),\qquad
\phi_{75}=\sin(7\pi x)\sin(5\pi y),
$$

the exact gradient is available analytically. This makes the $H^1$ diagnostic independent of automatic differentiation of the reference solution.


In [ ]:
# ============================================================
# MANUFACTURED TWO-FREQUENCY POISSON PROBLEM
# ============================================================
def phi11(x):
    """Low-frequency Dirichlet eigenfunction."""
    return torch.sin(math.pi * x[..., 0]) * torch.sin(math.pi * x[..., 1])


def phi75(x):
    """High-frequency Dirichlet eigenfunction."""
    return torch.sin(7.0 * math.pi * x[..., 0]) * torch.sin(5.0 * math.pi * x[..., 1])


def exact_solution(x):
    return phi11(x) + 0.1 * phi75(x)


def forcing(x):
    # -Delta(phi11)=2*pi^2*phi11 and -Delta(phi75)=74*pi^2*phi75.
    return 2.0 * math.pi**2 * phi11(x) + 7.4 * math.pi**2 * phi75(x)


def grad_phi11(x):
    gx = math.pi * torch.cos(math.pi * x[..., 0]) * torch.sin(math.pi * x[..., 1])
    gy = math.pi * torch.sin(math.pi * x[..., 0]) * torch.cos(math.pi * x[..., 1])
    return torch.stack((gx, gy), dim=-1)


def grad_phi75(x):
    gx = 7.0 * math.pi * torch.cos(7.0 * math.pi * x[..., 0]) * torch.sin(5.0 * math.pi * x[..., 1])
    gy = 5.0 * math.pi * torch.sin(7.0 * math.pi * x[..., 0]) * torch.cos(5.0 * math.pi * x[..., 1])
    return torch.stack((gx, gy), dim=-1)


def exact_gradient(x):
    return grad_phi11(x) + 0.1 * grad_phi75(x)


## 6. Sanity checks

The trial is $T_\theta(x)=\rho(x)N_\theta(x)$ with

$$
\rho(x,y)=\left(\frac32\right)^2(1-x^2)(1-y^2).
$$

Therefore $T_\theta=0$ and every parameter tangent $D_\theta T_\theta[d]=0$ on the boundary. We also compare the analytic forcing with $-\Delta u_*$ computed by automatic differentiation.


In [ ]:
# ============================================================
# REPRODUCIBLE MODEL CONSTRUCTION AND SANITY CHECKS
# ============================================================
def build_model_and_trial(seed, width, depth, activation):
    """Construct the common MLP family and its flat-parameter trial."""
    torch.manual_seed(seed)
    model = MLP(DIM, width=width, depth=depth, activation=activation).to(
        device=DEVICE, dtype=DTYPE
    )
    theta, spec = flatten_parameters(model)
    theta = theta.to(device=DEVICE, dtype=DTYPE)
    return model, theta, make_box_trial(model, spec)


_model_check, _theta_check, _trial_check = build_model_and_trial(
    PRIMARY_SEED,
    DTB_CFG["network_width"],
    DTB_CFG["network_depth"],
    DTB_CFG["activation"],
)

boundary_points = sample_box_boundary(
    32, DIM, seed=17, dtype=DTYPE, device=DEVICE
)
check_points = sample_box(64, DIM, seed=18, dtype=DTYPE, device=DEVICE)

max_trial_boundary = float(_trial_check(_theta_check, boundary_points).abs().max())
max_exact_boundary = float(exact_solution(boundary_points).abs().max())
automatic_f = manufactured_forcing(exact_solution, check_points)
forcing_difference = float((automatic_f - forcing(check_points)).abs().max())

print(f"max |trial| on boundary       = {max_trial_boundary:.3e}")
print(f"max |exact solution| boundary = {max_exact_boundary:.3e}")
print(f"max |analytic f - AD f|       = {forcing_difference:.3e}")

assert max_trial_boundary < 1.0e-12
assert max_exact_boundary < 1.0e-12
assert forcing_difference < 1.0e-8

del _model_check, _theta_check, _trial_check


## 7. Fixed train, test, residual, and plotting points

All methods use the same scrambled Sobol training quadrature and held-out points. Strong residuals require second derivatives, so they are evaluated on a fixed subset of the held-out points. This keeps the diagnostic reproducible without making every monitoring step prohibitively expensive.

For a uniformly sampled domain,

$$
\int_\Omega g(x)\,dx\approx \frac{|\Omega|}{N}\sum_{n=1}^N g(x_n).
$$


In [ ]:
# ============================================================
# SHARED POINT SETS
# ============================================================
x_train = sample_box(N_TRAIN, DIM, TRAIN_SEED, dtype=DTYPE, device=DEVICE)
x_test = sample_box(N_TEST, DIM, TEST_SEED, dtype=DTYPE, device=DEVICE)
x_residual = x_test[: min(N_RESIDUAL_TEST, N_TEST)]

f_train = forcing(x_train)
u_test = exact_solution(x_test)
grad_u_test = exact_gradient(x_test)

# A regular Cartesian grid is used only for plots, never for training.
plot_axis = torch.linspace(-1.0, 1.0, N_PLOT_SIDE, dtype=DTYPE, device=DEVICE)
plot_xx, plot_yy = torch.meshgrid(plot_axis, plot_axis, indexing="ij")
x_grid = torch.stack((plot_xx.reshape(-1), plot_yy.reshape(-1)), dim=-1)
u_grid_exact = exact_solution(x_grid).reshape(N_PLOT_SIDE, N_PLOT_SIDE).detach().cpu().numpy()

print("training points:", tuple(x_train.shape))
print("held-out points:", tuple(x_test.shape))
print("residual subset:", tuple(x_residual.shape))


## 8. Common diagnostics

Let $e=u_{\mathrm{pred}}-u_*$. Every method is evaluated with

$$
e_{L^2}=\frac{\|e\|_{L^2}}{\|u_*\|_{L^2}},\qquad
e_{H^1}=\frac{\|\nabla e\|_{L^2}}{\|\nabla u_*\|_{L^2}},
$$

$$
e_{\mathrm{res}}=\frac{\|-\Delta u_{\mathrm{pred}}-f\|_{L^2}}{\|f\|_{L^2}},
$$

and the Poisson energy gap

$$
\mathcal E(u_{\mathrm{pred}})-\mathcal E(u_*)
=\frac12\|\nabla(u_{\mathrm{pred}}-u_*)\|_{L^2}^2.
$$

The recovered high-mode coefficient is

$$
\widehat a_{75}=\int_\Omega u_{\mathrm{pred}}\phi_{75}\,dx.
$$

Here $\|\phi_{75}\|_{L^2(\Omega)}^2=1$, so the target value is exactly $0.1$.


In [ ]:
# ============================================================
# SHARED EVALUATION HELPERS
# ============================================================
def synchronize():
    """Make wall-clock timings meaningful when CUDA is asynchronous."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def neural_values_and_gradients(trial, theta, points, chunk_size=512):
    """Evaluate a direct neural trial and its spatial gradient in chunks."""
    def value_one(point):
        return trial(theta, point)

    gradient_one = jacrev(value_one)
    values, gradients = [], []
    for start in range(0, points.shape[0], chunk_size):
        chunk = points[start : start + chunk_size]
        values.append(vmap(value_one)(chunk))
        gradients.append(vmap(gradient_one)(chunk))
    return torch.cat(values), torch.cat(gradients)


def neural_negative_laplacian(trial, theta, points, chunk_size=128):
    """Evaluate -Delta T_theta using exact automatic differentiation."""
    def value_one(point):
        return trial(theta, point)

    hessian_one = jacrev(jacrev(value_one))
    pieces = []
    for start in range(0, points.shape[0], chunk_size):
        hessian = vmap(hessian_one)(points[start : start + chunk_size])
        pieces.append(-torch.diagonal(hessian, dim1=-2, dim2=-1).sum(-1))
    return torch.cat(pieces)


def tangent_negative_laplacian(trial, theta, direction, points, chunk_size=128):
    """Evaluate -Delta(D_theta T_theta[direction]) without a full parameter Jacobian."""
    def value_one(point):
        return jvp(
            lambda candidate: trial(candidate, point),
            (theta,),
            (direction,),
        )[1]

    hessian_one = jacrev(jacrev(value_one))
    pieces = []
    for start in range(0, points.shape[0], chunk_size):
        hessian = vmap(hessian_one)(points[start : start + chunk_size])
        pieces.append(-torch.diagonal(hessian, dim1=-2, dim2=-1).sum(-1))
    return torch.cat(pieces)


def common_metrics(values, gradients, negative_laplacian=None):
    """Compute common held-out metrics from values and first derivatives."""
    value_error = values - u_test
    gradient_error = gradients - grad_u_test

    l2 = torch.linalg.norm(value_error) / torch.linalg.norm(u_test)
    h1 = torch.linalg.norm(gradient_error) / torch.linalg.norm(grad_u_test)
    energy_gap = 0.5 * VOLUME * torch.mean(torch.sum(gradient_error.square(), dim=-1))
    a75 = VOLUME * torch.mean(values * phi75(x_test))

    if negative_laplacian is None:
        residual = torch.tensor(float("nan"), dtype=DTYPE, device=DEVICE)
    else:
        f_residual = forcing(x_residual)
        residual = torch.linalg.norm(negative_laplacian - f_residual) / torch.linalg.norm(f_residual)

    return dict(
        l2=float(l2),
        h1=float(h1),
        residual=float(residual),
        energy=float(energy_gap),
        a75=float(a75),
    )


def tangent_metrics(trial, theta, direction, include_residual=False):
    values, gradients = tangent_values_and_gradients(
        trial, theta, direction, x_test, chunk_size=512
    )
    negative_laplacian = None
    if include_residual:
        negative_laplacian = tangent_negative_laplacian(
            trial, theta, direction, x_residual
        )
    return common_metrics(values, gradients, negative_laplacian)


def direct_neural_metrics(trial, theta, include_residual=False):
    values, gradients = neural_values_and_gradients(trial, theta, x_test)
    negative_laplacian = None
    if include_residual:
        negative_laplacian = neural_negative_laplacian(trial, theta, x_residual)
    return common_metrics(values, gradients, negative_laplacian)


def mode_projection_error(feature_gradients, target_gradients):
    """Relative H1-seminorm projection error onto the selected tangent space."""
    design = feature_gradients.reshape(-1, feature_gradients.shape[-1])
    target = target_gradients.reshape(-1)

    gram = design.T @ design
    rhs = design.T @ target
    scale = (torch.trace(gram) / gram.shape[0]).clamp_min(torch.finfo(DTYPE).eps)
    beta = torch.linalg.solve(
        gram + 1.0e-10 * scale * torch.eye(gram.shape[0], dtype=DTYPE, device=DEVICE),
        rhs,
    )
    return float(torch.linalg.norm(design @ beta - target) / torch.linalg.norm(target))


def low_mode_projection_error(feature_gradients):
    """H1 projection error of phi11, the low-frequency solution component."""
    return mode_projection_error(feature_gradients, grad_phi11(x_train))


def high_mode_projection_error(feature_gradients):
    """H1 projection error of phi75, retained for the original diagnostic."""
    return mode_projection_error(feature_gradients, grad_phi75(x_train))


## 9. Results container

All runners write to one dictionary. Plotting cells only read this dictionary, so rerunning a figure never retrains a model.


In [ ]:
# ============================================================
# RESULTS CONTAINER
# ============================================================
results = {
    "dtb": defaultdict(dict),
    "deep_ritz": {},
    "fourier": {},
    "fourier_oracle": None,
}


## 10. Method A: adaptive DTB--Ritz

At parameters $\theta$, select $r$ parameter coordinates $I$ and define tangent features

$$
J_i^\theta(x)=\frac{\partial T_\theta(x)}{\partial\theta_{I_i}},\qquad
u_{\theta,\alpha}(x)=\sum_{i=1}^r\alpha_iJ_i^\theta(x).
$$

The inner Ritz problem is

$$
\alpha_\theta=\arg\min_{\alpha\in\mathbb R^r}
\left\{\frac12\alpha^\top G_\theta\alpha-b_\theta^\top\alpha\right\},
$$

where

$$
(G_\theta)_{ij}=\int_\Omega\nabla J_i^\theta\cdot\nabla J_j^\theta\,dx,
\qquad (b_\theta)_i=\int_\Omega fJ_i^\theta\,dx.
$$

After solving the small $r\times r$ system, the outer step differentiates the unregularized energy while holding the solved tangent direction fixed. Define the descent velocity and fixed-step update by

$$
\dot\theta_k=-\nabla_\theta E(\theta_k,\alpha_k),\qquad
\theta_{k+1}=\theta_k+\eta\dot\theta_k,\qquad \eta=10^{-2}.
$$

No line search or Armijo backtracking is used. The notebook records $\|\dot\theta_k\|_2$ at each monitored iteration.

For a mode $\phi_{mn}$, the DTB projection diagnostic is

$$
\varepsilon_{mn}^{\mathrm{proj}}(\theta)=
\frac{\|\nabla(\phi_{mn}-\Pi_{V_r(\theta)}\phi_{mn})\|_{L^2}}
{\|\nabla\phi_{mn}\|_{L^2}}.
$$


In [ ]:
# ============================================================
# DTB RUNNER
# ============================================================
def run_dtb(rank, seed, cfg, verbose=True):
    """Run one frozen-to-adaptive DTB experiment and retain all diagnostics."""
    model, theta, trial = build_model_and_trial(
        seed,
        cfg["network_width"],
        cfg["network_depth"],
        cfg["activation"],
    )
    indices = select_parameter_indices(
        theta.numel(), rank, cfg["tangent_seed"], device=DEVICE
    )

    checkpoints = set(cfg["checkpoints"])
    history = []
    snapshot_predictions = {}

    synchronize()
    start_time = time.perf_counter()

    for step in range(cfg["n_outer"] + 1):
        # Block A: construct the selected tangent basis J and grad J.
        features, feature_gradients = selected_tangent_basis(
            trial,
            theta,
            x_train,
            indices,
            chunk_size=cfg["chunk_size"],
        )

        # Block B: assemble G and b, then solve the stabilized inner Ritz system.
        stiffness, load = assemble_ritz_system(
            features,
            feature_gradients,
            f_train,
            VOLUME,
            diffusion=DIFFUSION,
            reaction=REACTION,
        )
        solution = solve_ritz_system(
            stiffness, load, ridge_relative=cfg["ridge_relative"]
        )
        direction = expand_direction(solution.alpha, indices, theta.numel())

        # Block C: compute the fixed-step outer descent velocity dot(theta).
        outer_gradient = envelope_gradient(
            trial,
            theta,
            direction,
            x_train,
            f_train,
            VOLUME,
            diffusion=DIFFUSION,
            reaction=REACTION,
        )
        theta_dot = -outer_gradient.detach()

        # Block D: monitor held-out accuracy and tangent representation quality.
        should_monitor = (
            step % MONITOR_EVERY == 0
            or step in checkpoints
            or step == cfg["n_outer"]
        )
        if should_monitor:
            include_residual = step in checkpoints or step == cfg["n_outer"]
            metrics = tangent_metrics(
                trial, theta, direction, include_residual=include_residual
            )
            synchronize()
            row = dict(
                step=step,
                wall_time=time.perf_counter() - start_time,
                F_theta=float(matrix_ritz_energy(stiffness, load, solution.alpha)),
                proj11=low_mode_projection_error(feature_gradients),
                proj75=high_mode_projection_error(feature_gradients),
                alpha_norm=float(torch.linalg.norm(solution.alpha)),
                theta_dot_norm=float(torch.linalg.norm(theta_dot)),
                inner_residual=solution.relative_residual,
                ridge_shift=solution.ridge_shift,
                fixed_step_size=cfg["fixed_step_size"],
                **metrics,
            )
            history.append(row)

        # Block E: retain only the requested grid snapshots.
        if step in checkpoints:
            prediction = []
            for start in range(0, x_grid.shape[0], 1024):
                chunk = x_grid[start : start + 1024]
                value = jvp(
                    lambda candidate: trial(candidate, chunk),
                    (theta,),
                    (direction,),
                )[1]
                prediction.append(value.detach())
            snapshot_predictions[step] = (
                torch.cat(prediction)
                .reshape(N_PLOT_SIDE, N_PLOT_SIDE)
                .cpu()
                .numpy()
            )

        if step == cfg["n_outer"]:
            break

        # Block F: fixed explicit update; no line search is performed.
        theta = (theta + cfg["fixed_step_size"] * theta_dot).detach()

    record = dict(
        rank=rank,
        seed=seed,
        history=history,
        checkpoints=snapshot_predictions,
        final_wall_time=history[-1]["wall_time"],
    )

    if verbose:
        for row in (history[0], history[len(history) // 2], history[-1]):
            print(
                f"DTB step={row['step']:3d}  "
                f"L2={row['l2']:.3e}  H1={row['h1']:.3e}  "
                f"proj11={row['proj11']:.3e}  |dot(theta)|={row['theta_dot_norm']:.3e}"
            )

    return record


In [ ]:
# ============================================================
# PRIMARY DTB RUN: r = 32, seed = 0 in the full configuration
# ============================================================
dtb_primary = run_dtb(PRIMARY_RANK, PRIMARY_SEED, DTB_CFG)
results["dtb"][PRIMARY_RANK][PRIMARY_SEED] = dtb_primary


## 11. Method B: ordinary Deep Ritz

Ordinary Deep Ritz trains the full hard-Dirichlet neural trial

$$
u_\theta(x)=\rho(x)N_\theta(x)
$$

by minimizing

$$
\mathcal E(\theta)=\frac12\int_\Omega|\nabla u_\theta|^2\,dx
-\int_\Omega fu_\theta\,dx.
$$

Unlike DTB, this baseline optimizes all $p$ network parameters directly. It uses the same architecture, initial seed, boundary factor, and training quadrature, but its Adam steps are not equated with DTB outer iterations.


In [ ]:
# ============================================================
# ORDINARY DEEP RITZ RUNNER
# ============================================================
def run_deep_ritz(seed, cfg, verbose=True):
    """Train every flat MLP parameter with the standard Ritz energy."""
    model, theta_initial, trial = build_model_and_trial(
        seed,
        cfg["network_width"],
        cfg["network_depth"],
        cfg["activation"],
    )
    theta = torch.nn.Parameter(theta_initial.clone())
    optimizer = torch.optim.Adam(
        [theta],
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
    )

    checkpoint_steps = {0, cfg["n_steps"] // 2, cfg["n_steps"]}
    history = []
    snapshots = {}

    synchronize()
    start_time = time.perf_counter()

    for step in range(cfg["n_steps"] + 1):
        # Block A: monitor before the next optimizer update.
        should_monitor = step % cfg["monitor_every"] == 0 or step in checkpoint_steps
        if should_monitor:
            theta_eval = theta.detach()
            include_residual = step in checkpoint_steps
            metrics = direct_neural_metrics(
                trial, theta_eval, include_residual=include_residual
            )
            train_values, train_gradients = neural_values_and_gradients(
                trial, theta_eval, x_train
            )
            train_energy = VOLUME * torch.mean(
                0.5 * torch.sum(train_gradients.square(), dim=-1)
                - f_train * train_values
            )
            synchronize()
            history.append(
                dict(
                    step=step,
                    wall_time=time.perf_counter() - start_time,
                    train_energy=float(train_energy),
                    **metrics,
                )
            )

        # Block B: retain the same diagnostic-time snapshots as DTB.
        if step in checkpoint_steps:
            values, _ = neural_values_and_gradients(trial, theta.detach(), x_grid)
            snapshots[step] = (
                values.detach()
                .reshape(N_PLOT_SIDE, N_PLOT_SIDE)
                .cpu()
                .numpy()
            )

        if step == cfg["n_steps"]:
            break

        # Block C: one ordinary full-parameter Adam update.
        optimizer.zero_grad(set_to_none=True)
        train_values, train_gradients = neural_values_and_gradients(
            trial, theta, x_train
        )
        loss = VOLUME * torch.mean(
            0.5 * torch.sum(train_gradients.square(), dim=-1)
            - f_train * train_values
        )
        loss.backward()
        optimizer.step()

    record = dict(
        seed=seed,
        history=history,
        checkpoints=snapshots,
        final_wall_time=history[-1]["wall_time"],
    )

    if verbose:
        for row in (history[0], history[len(history) // 2], history[-1]):
            print(
                f"Deep Ritz step={row['step']:5d}  "
                f"L2={row['l2']:.3e}  H1={row['h1']:.3e}  a75={row['a75']:.3e}"
            )

    return record


In [ ]:
# ============================================================
# ORDINARY DEEP RITZ PRIMARY RUN
# ============================================================
deep_ritz_primary = run_deep_ritz(PRIMARY_SEED, DEEP_RITZ_CFG)
results["deep_ritz"][PRIMARY_SEED] = deep_ritz_primary


## 12. Method C: complete Fourier/Dirichlet Ritz basis

The complete Dirichlet eigenbasis on $(-1,1)^2$ is

$$
\psi_{mn}(x,y)=
\sin\!\left(\frac{m\pi(x+1)}2\right)
\sin\!\left(\frac{n\pi(y+1)}2\right),\qquad m,n\geq1,
$$

with

$$
-\Delta\psi_{mn}=\lambda_{mn}\psi_{mn},\qquad
\lambda_{mn}=\frac{\pi^2}{4}(m^2+n^2).
$$

Modes are ordered by increasing $m^2+n^2$. The low target mode is $\psi_{2,2}$ and the high target mode is $\psi_{14,10}$. The latter appears only at spectral position 217 or 218, so no truncation with $r\leq64$ contains it exactly.

For the first $r$ modes, solve

$$
Gc=b,\qquad G_{ij}=\int_\Omega\nabla\psi_i\cdot\nabla\psi_j\,dx,
\qquad b_i=\int_\Omega f\psi_i\,dx.
$$


In [ ]:
# ============================================================
# FOURIER / DIRICHLET RITZ SOLVER
# ============================================================
def ordered_dirichlet_modes(rank):
    """Return the first rank pairs (m,n), ordered by eigenvalue then lexically."""
    maximum_index = max(16, 2 * math.ceil(math.sqrt(rank)) + 4)
    pairs = [
        (m, n)
        for m in range(1, maximum_index + 1)
        for n in range(1, maximum_index + 1)
    ]
    pairs.sort(key=lambda pair: (pair[0] ** 2 + pair[1] ** 2, pair[0], pair[1]))
    return pairs[:rank]


def fourier_basis_fields(points, modes):
    """Analytic values, gradients, and eigenvalues for the shifted sine basis."""
    m = torch.tensor([pair[0] for pair in modes], dtype=DTYPE, device=DEVICE)
    n = torch.tensor([pair[1] for pair in modes], dtype=DTYPE, device=DEVICE)

    ax = 0.5 * math.pi * (points[:, 0:1] + 1.0) * m
    ay = 0.5 * math.pi * (points[:, 1:2] + 1.0) * n
    sx, cx = torch.sin(ax), torch.cos(ax)
    sy, cy = torch.sin(ay), torch.cos(ay)

    values = sx * sy
    grad_x = (0.5 * math.pi * m) * cx * sy
    grad_y = (0.5 * math.pi * n) * sx * cy
    gradients = torch.stack((grad_x, grad_y), dim=1)
    eigenvalues = 0.25 * math.pi**2 * (m.square() + n.square())
    return values, gradients, eigenvalues


def solve_fourier_rank(rank):
    """Solve one nonadaptive spectral Ritz problem using the shared points."""
    modes = ordered_dirichlet_modes(rank)

    # Block A: assemble the same empirical Ritz system used by the neural methods.
    train_values, train_gradients, _ = fourier_basis_fields(x_train, modes)
    stiffness, load = assemble_ritz_system(
        train_values,
        train_gradients,
        f_train,
        VOLUME,
        diffusion=DIFFUSION,
        reaction=REACTION,
    )
    solution = solve_ritz_system(
        stiffness,
        load,
        ridge_relative=FOURIER_CFG["ridge_relative"],
    )

    # Block B: evaluate values and gradients analytically on the held-out set.
    test_values, test_gradients, test_eigenvalues = fourier_basis_fields(x_test, modes)
    prediction = test_values @ solution.alpha
    prediction_gradient = torch.einsum("ndr,r->nd", test_gradients, solution.alpha)

    # Block C: the strong residual is also analytic because every basis function is an eigenmode.
    residual_values, _, residual_eigenvalues = fourier_basis_fields(x_residual, modes)
    negative_laplacian = residual_values @ (residual_eigenvalues * solution.alpha)
    metrics = common_metrics(prediction, prediction_gradient, negative_laplacian)

    grid_values, _, _ = fourier_basis_fields(x_grid, modes)
    grid_prediction = (
        (grid_values @ solution.alpha)
        .detach()
        .reshape(N_PLOT_SIDE, N_PLOT_SIDE)
        .cpu()
        .numpy()
    )

    return dict(
        rank=rank,
        modes=modes,
        coefficients=solution.alpha.detach().cpu().numpy(),
        prediction=grid_prediction,
        **metrics,
    )


# Solve every requested Fourier rank. These solves are small and non-iterative.
for rank in FOURIER_RANKS:
    results["fourier"][rank] = solve_fourier_rank(rank)
    row = results["fourier"][rank]
    print(
        f"Fourier r={rank:3d}  L2={row['l2']:.3e}  "
        f"H1={row['h1']:.3e}  a75={row['a75']:.3e}"
    )


### Optional oracle Fourier check

The two-dimensional oracle space $\operatorname{span}\{\phi_{11},\phi_{75}\}$ uses prior knowledge of the true active frequencies. It should recover coefficients $(1,0.1)$ nearly exactly, but it is not a fair nonadaptive baseline.


In [ ]:
# ============================================================
# ORACLE FOURIER SANITY CHECK (NOT A FAIR BASELINE)
# ============================================================
if RUN_ORACLE_FOURIER:
    oracle_modes = [(2, 2), (14, 10)]

    # These two eigenfunctions are exactly L2-orthonormal on (-1,1)^2.
    # Therefore G is diagonal with the two Laplacian eigenvalues, while
    # b contains each eigenvalue multiplied by the exact modal coefficient.
    oracle_eigenvalues = torch.tensor(
        [2.0 * math.pi**2, 74.0 * math.pi**2],
        dtype=DTYPE,
        device=DEVICE,
    )
    stiffness = torch.diag(oracle_eigenvalues)
    load = oracle_eigenvalues * torch.tensor(
        [1.0, 0.1], dtype=DTYPE, device=DEVICE
    )
    oracle_solution = solve_ritz_system(stiffness, load, ridge_relative=0.0)

    test_values, test_gradients, _ = fourier_basis_fields(x_test, oracle_modes)
    prediction = test_values @ oracle_solution.alpha
    prediction_gradient = torch.einsum(
        "ndr,r->nd", test_gradients, oracle_solution.alpha
    )
    residual_values, _, residual_eigenvalues = fourier_basis_fields(
        x_residual, oracle_modes
    )
    negative_laplacian = residual_values @ (
        residual_eigenvalues * oracle_solution.alpha
    )
    results["fourier_oracle"] = dict(
        coefficients=oracle_solution.alpha.detach().cpu().numpy(),
        **common_metrics(prediction, prediction_gradient, negative_laplacian),
    )
    print("oracle coefficients:", results["fourier_oracle"]["coefficients"])
    print("oracle relative L2:", results["fourier_oracle"]["l2"])


## 13. Figure 1: aligned reference, Deep Ritz, and DTB snapshots

Rows show the reference solution, ordinary Deep Ritz, and DTB. Columns align the initial, midpoint, and final diagnostic stages. Every panel uses the same spatial extent, axes, and color scale.

The columns align stages for visual comparison only. The annotations retain the actual Deep Ritz Adam step and DTB outer step; they must not be interpreted as equal computational work.


In [ ]:
# ============================================================
# FIGURE 1: ALIGNED REFERENCE, DEEP RITZ, AND DTB SNAPSHOTS
# ============================================================
dtb_snapshot_items = sorted(dtb_primary["checkpoints"].items())
deep_ritz_snapshot_items = sorted(deep_ritz_primary["checkpoints"].items())
stage_labels = ["initial", "midpoint", "final"]

assert len(dtb_snapshot_items) == len(stage_labels)
assert len(deep_ritz_snapshot_items) == len(stage_labels)

# One shared scale is essential: otherwise a weak prediction can look
# artificially as strong as the reference after per-panel normalization.
all_snapshot_fields = [u_grid_exact]
all_snapshot_fields += [field for _, field in deep_ritz_snapshot_items]
all_snapshot_fields += [field for _, field in dtb_snapshot_items]
color_limit = max(float(np.max(np.abs(field))) for field in all_snapshot_fields)

fig, axes = plt.subplots(
    3,
    3,
    figsize=(12.5, 10.0),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for column, stage in enumerate(stage_labels):
    deep_step, deep_field = deep_ritz_snapshot_items[column]
    dtb_step, dtb_field = dtb_snapshot_items[column]

    aligned_fields = [u_grid_exact, deep_field, dtb_field]
    annotations = [r"$u_*$", f"Adam step {deep_step}", f"outer step {dtb_step}"]

    for row, (field, annotation) in enumerate(zip(aligned_fields, annotations)):
        image = axes[row, column].imshow(
            field.T,
            origin="lower",
            extent=(-1, 1, -1, 1),
            cmap="coolwarm",
            vmin=-color_limit,
            vmax=color_limit,
        )
        axes[row, column].text(
            0.04,
            0.94,
            annotation,
            transform=axes[row, column].transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=2.0),
        )

    axes[0, column].set_title(stage.capitalize())
    axes[2, column].set_xlabel("x")

axes[0, 0].set_ylabel("Reference\ny")
axes[1, 0].set_ylabel("Ordinary Deep Ritz\ny")
axes[2, 0].set_ylabel("DTB\ny")

fig.colorbar(
    image,
    ax=axes.ravel().tolist(),
    shrink=0.78,
    pad=0.02,
    label="solution value",
)
fig.suptitle("Figure 1: aligned solution snapshots", fontsize=15)
plt.show()


## 14. Figure 2: DTB error evolution

A gap between $e_{L^2}$ and $e_{H^1}$ indicates that coarse function values have been learned before the oscillatory slopes.


In [ ]:
# ============================================================
# FIGURE 2: DTB L2 AND H1 ERROR CURVES
# ============================================================
dtb_history = dtb_primary["history"]
dtb_steps = [row["step"] for row in dtb_history]

plt.figure(figsize=(7.0, 4.4))
plt.semilogy(dtb_steps, [row["l2"] for row in dtb_history], "o-", label=r"relative $L^2$")
plt.semilogy(dtb_steps, [row["h1"] for row in dtb_history], "s-", label=r"relative $H^1$ seminorm")
plt.xlabel("DTB outer iteration")
plt.ylabel("relative error")
plt.title("Figure 2: DTB error evolution")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 15. Figure 3: DTB representation adaptation

The two panels distinguish capacity from realization:

$$
\varepsilon_{75}^{\mathrm{proj}}\downarrow
\quad\Longrightarrow\quad
\text{the tangent space can represent }\phi_{75},
$$

while $\widehat a_{75}\to0.1$ means the solved DTB prediction actually uses that capacity.


In [ ]:
# ============================================================
# FIGURE 3: PROJECTION ERROR AND RECOVERED HIGH-MODE COEFFICIENT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].semilogy(dtb_steps, [row["proj75"] for row in dtb_history], "o-")
axes[0].set(
    xlabel="DTB outer iteration",
    ylabel=r"$\varepsilon_{75}^{\mathrm{proj}}$",
    title="Tangent-space projection error",
)
axes[0].grid(True, which="both", alpha=0.3)

axes[1].plot(dtb_steps, [row["a75"] for row in dtb_history], "o-", label=r"$\widehat a_{75}$")
axes[1].axhline(0.1, color="black", linestyle="--", label="target 0.1")
axes[1].set(
    xlabel="DTB outer iteration",
    ylabel="coefficient",
    title="Recovered high-frequency coefficient",
)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle("Figure 3: DTB representation adaptation", y=1.03)
plt.tight_layout()
plt.show()


## 16. Figure 4: DTB error heatmaps

All heatmaps use one symmetric scale. If the initial tangent space misses the high mode, the early error should visually resemble $-0.1\phi_{75}$.


In [ ]:
# ============================================================
# FIGURE 4: SIGNED DTB ERROR AT THE THREE CHECKPOINTS
# ============================================================
error_fields = [
    (step, prediction - u_grid_exact)
    for step, prediction in dtb_snapshot_items
]
error_limit = max(float(np.max(np.abs(field))) for _, field in error_fields)

fig, axes = plt.subplots(1, len(error_fields), figsize=(4.4 * len(error_fields), 3.8), sharex=True, sharey=True)
if len(error_fields) == 1:
    axes = [axes]
for ax, (step, field) in zip(axes, error_fields):
    image = ax.imshow(
        field.T,
        origin="lower",
        extent=(-1, 1, -1, 1),
        cmap="coolwarm",
        vmin=-error_limit,
        vmax=error_limit,
    )
    ax.set_title(f"k={step}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.colorbar(image, ax=axes, shrink=0.78, label=r"$u_{\mathrm{DTB}}-u_*$")
fig.suptitle("Figure 4: DTB signed-error evolution", y=1.03)
plt.show()


## 17. Figure 5: final method comparison

Frozen DTB is the solved tangent approximation at $k=0$. Adaptive DTB is the final outer iterate. Ordinary Deep Ritz is reported at its final Adam step, and Fourier Ritz uses the same selected rank as the primary DTB run.


In [ ]:
# ============================================================
# FIGURE 5: COMMON FINAL METRICS AND WALL-CLOCK SUMMARY
# ============================================================
frozen_dtb = dtb_primary["history"][0]
adaptive_dtb = dtb_primary["history"][-1]
deep_ritz_final = deep_ritz_primary["history"][-1]
fourier_final = results["fourier"][PRIMARY_RANK]

method_rows = [
    ("frozen DTB", frozen_dtb),
    ("adaptive DTB", adaptive_dtb),
    ("ordinary Deep Ritz", deep_ritz_final),
    (f"Fourier r={PRIMARY_RANK}", fourier_final),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metric_specs = [
    ("l2", r"relative $L^2$", True),
    ("h1", r"relative $H^1$", True),
    ("residual", "strong residual", True),
    ("a75", r"$\widehat a_{75}$", False),
]
labels = [name for name, _ in method_rows]

for ax, (key, title, logarithmic) in zip(axes.flat, metric_specs):
    values = [row[key] for _, row in method_rows]
    ax.bar(np.arange(len(labels)), values)
    ax.set_xticks(np.arange(len(labels)), labels, rotation=20, ha="right")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    if logarithmic:
        ax.set_yscale("log")
    if key == "a75":
        ax.axhline(0.1, color="black", linestyle="--", label="target 0.1")
        ax.legend()

fig.suptitle("Figure 5: final method comparison")
plt.tight_layout()
plt.show()

print(f"{'method':24s} {'L2':>11s} {'H1':>11s} {'residual':>11s} {'a75':>11s} {'seconds':>11s}")
print("-" * 84)
for name, row in method_rows:
    if name == "frozen DTB":
        seconds = row["wall_time"]
    elif name == "adaptive DTB":
        seconds = dtb_primary["final_wall_time"]
    elif name == "ordinary Deep Ritz":
        seconds = deep_ritz_primary["final_wall_time"]
    else:
        seconds = float("nan")
    print(
        f"{name:24s} {row['l2']:11.3e} {row['h1']:11.3e} "
        f"{row['residual']:11.3e} {row['a75']:11.3e} {seconds:11.2f}"
    )


## 18. DTB-only tangent-basis comparison

This block runs DTB with $r\in\{8,32,64\}$ using the same initial network, selected-coordinate seed, quadrature points, fixed step $\eta=10^{-2}$, and number of outer iterations. The already-computed primary run is reused. No Deep Ritz or Fourier result enters this comparison. Set `RUN_TANGENT_COMPARISON=False` only when you want to skip the two additional DTB runs.

The optional seed sweep remains separate and is disabled by default.


In [ ]:
# ============================================================
# DTB-ONLY TANGENT-SIZE COMPARISON AND OPTIONAL SEED SWEEP
# ============================================================
if RUN_TANGENT_COMPARISON:
    for rank in TANGENT_RANKS:
        if PRIMARY_SEED not in results["dtb"][rank]:
            rank_cfg = dict(DTB_CFG, tangent_rank=rank)
            results["dtb"][rank][PRIMARY_SEED] = run_dtb(
                rank, PRIMARY_SEED, rank_cfg, verbose=False
            )
    print(f"DTB tangent-size comparison complete: {TANGENT_RANKS}")
else:
    print("Tangent-size comparison skipped.")

if RUN_SEED_SWEEP:
    for seed in SEEDS:
        if seed not in results["dtb"][PRIMARY_RANK]:
            results["dtb"][PRIMARY_RANK][seed] = run_dtb(
                PRIMARY_RANK, seed, DTB_CFG, verbose=False
            )
    print("DTB seed sweep complete.")
else:
    print("Seed sweep skipped. Set RUN_SEED_SWEEP=True to enable it.")


## 19. Figure 6: DTB trajectories across tangent-basis sizes

Every curve below is a DTB run; no other solver is shown. The horizontal coordinate is elapsed wall-clock time for that run. The first two panels use

$$
e_{L^2}(t)=\frac{\|u_{\theta(t)}-u_*\|_{L^2}}{\|u_*\|_{L^2}},\qquad
e_{H^1}(t)=\frac{\|\nabla u_{\theta(t)}-\nabla u_*\|_{L^2}}{\|\nabla u_*\|_{L^2}}.
$$

The lower panels show the low-frequency $H^1$ projection error

$$
\varepsilon_{11}^{\mathrm{proj}}(t)=
\frac{\|\nabla(\phi_{11}-\Pi_{V_r(\theta(t))}\phi_{11})\|_{L^2}}
{\|\nabla\phi_{11}\|_{L^2}},
$$

and the parameter-space descent speed $\|\dot\theta(t)\|_2$. Because parameter norms depend on the chosen neural parameterization, this final panel is a within-architecture diagnostic; all three runs use the same architecture and initialization.


In [ ]:
# ============================================================
# FIGURE 6: DTB-ONLY TRAJECTORIES ACROSS TANGENT-BASIS SIZES
# ============================================================
def plot_tangent_size_trajectories(records, ranks, seed, x_key="wall_time"):
    """Compare any stored DTB trajectory metrics across tangent sizes."""
    metric_specs = [
        ("l2", r"relative $L^2$ error"),
        ("h1", r"relative $H^1$ seminorm error"),
        ("proj11", r"$\varepsilon_{11}^{\mathrm{proj}}$"),
        ("theta_dot_norm", r"$\|\dot\theta\|_2$"),
    ]
    x_labels = {"wall_time": "elapsed wall time (s)", "step": "DTB outer iteration"}
    if x_key not in x_labels:
        raise ValueError(f"x_key must be one of {tuple(x_labels)}")

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2))
    for rank in ranks:
        history = records["dtb"][rank][seed]["history"]
        x_values = [row[x_key] for row in history]
        for ax, (metric, ylabel) in zip(axes.flat, metric_specs):
            ax.semilogy(
                x_values,
                [row[metric] for row in history],
                linewidth=2.0,
                label=fr"$r={rank}$",
            )

    titles = [
        r"Solution error in $L^2$",
        r"Gradient error in $H^1$ seminorm",
        r"Low-frequency tangent projection",
        r"Fixed-step parameter velocity",
    ]
    for ax, (_, ylabel), title in zip(axes.flat, metric_specs, titles):
        ax.set(xlabel=x_labels[x_key], ylabel=ylabel, title=title)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()

    fig.suptitle(
        fr"Figure 6: DTB tangent-size trajectories ($\eta={DTB_CFG['fixed_step_size']:.0e}$)",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


if RUN_TANGENT_COMPARISON:
    plot_tangent_size_trajectories(
        results, TANGENT_RANKS, PRIMARY_SEED, x_key="wall_time"
    )
    # For iteration-aligned curves, rerun with x_key="step".
else:
    print("Figure 6 is waiting for RUN_TANGENT_COMPARISON=True.")


## 20. Interpretation checklist

The strongest evidence for useful DTB adaptation is the chain

$$
\varepsilon_{75}^{\mathrm{proj}}(\theta_k)\downarrow
\quad\Longrightarrow\quad
\widehat a_{75}(k)\to0.1
\quad\Longrightarrow\quad
e_{H^1}(k)\downarrow.
$$

- Frozen DTB vs. adaptive DTB asks whether changing $\theta$ improves the tangent representation at fixed $r$.
- Adaptive DTB vs. Fourier-$r$ asks whether a learned $r$-dimensional tangent space allocates capacity more problem-adaptively than the first $r$ fixed spectral modes.
- Adaptive DTB vs. ordinary Deep Ritz compares the DTB formulation with direct full-network energy minimization.
- The DTB-only tangent-size trajectories distinguish adaptation at fixed $r$ from improvement caused by increasing tangent dimension.
- $\varepsilon_{11}^{\mathrm{proj}}$ measures whether the current tangent space can represent the low-frequency mode; it is not the solution error itself.
- $\|\dot\theta\|_2$ measures the requested fixed-step descent velocity. The actual increment has norm $\eta\|\dot\theta\|_2$.

A small $L^2$ error with a larger $H^1$ error means coarse function values are already accurate while oscillatory slopes are not. A large strong residual is the direct indication that the second-derivative structure required by the PDE is still missing. Because the fixed update performs no acceptance test, its energy and error curves are not guaranteed to decrease monotonically.
